In [1]:
import os
import time
import numpy as np
from collections.abc import Callable
import importlib

import torch
import torch.nn as nn
from torch.amp import autocast
from huggingface_hub import hf_hub_download

os.chdir("/home/pkulkarni/maisi-vae-multigpu")
from src.extras.utils import *

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Count:", torch.cuda.device_count())

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


CUDA available: True
GPU: NVIDIA RTX A6000
Count: 8


In [2]:
# Download MAISI VAE weights
model_path = hf_hub_download(
    repo_id="nvidia/NV-Generate-CT",
    filename="models/autoencoder_v1.pt",
)

# Download example image from CT-RATE
# (https://huggingface.co/datasets/ibrahimhamamci/CT-RATE)
img_path = hf_hub_download(
    repo_id="ibrahimhamamci/CT-RATE",
    filename="dataset/valid_fixed/valid_1/valid_1_a/valid_1_a_1.nii.gz",
    repo_type="dataset",
)

In [5]:
version = 2
num_splits = 2
num_devices = 2
save_mem = True
device = "cuda"

img = load_image(img_path, [512, 512, 192]).to(device)
print(img.shape)

model = load_vae_model(model_path, version, num_splits, num_devices, device)

monai.transforms.spatial.array Orientation.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


torch.Size([1, 1, 512, 512, 192])


In [7]:
start_time = time.time()

with torch.no_grad(), autocast("cuda"):
    _ = model.encode_stage_2_inputs(img)
    
end_time = time.time()
print(end_time - start_time)

6.222944736480713
